# EURUSD at one second

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adona-Robot/market-data-examples/blob/main/notebooks/eurusd_1s.ipynb)

Reads the last hour of EURUSD one-second candles from the [adona-robot](https://adona-robot.com/en) API, as a pandas DataFrame, and draws it.

You need an API key: the [free trial](https://adona-robot.com/en/signup) carries EURUSD and XAUUSD with 7 days of history. Every page read is billed in slices of 1 000 bars from your plan; this notebook reads about four.

In [ ]:
%pip install -q "adona-market-data[pandas]" matplotlib

In [ ]:
import os
from getpass import getpass

# The API key, from Colab's secrets (key icon on the left, name ADONA_API_KEY) or typed here.
# It stays in this session: never paste it into a cell's text.
try:
    from google.colab import userdata
    os.environ["ADONA_API_KEY"] = userdata.get("ADONA_API_KEY")
except Exception:
    if not os.environ.get("ADONA_API_KEY"):
        os.environ["ADONA_API_KEY"] = getpass("API key (adk_...): ")

## The last hour

`candles()` pages back with the API's cursor and returns the bars oldest first. A one-second bar exists only for a second in which the price was updated, so a quiet minute has fewer than sixty.

In [ ]:
from adona_market_data import Client

client = Client(end_user_id="colab")
bars = client.candles("EURUSD", "S1", bars=3600)
frame = client.to_pandas(bars)
frame.tail()

In [ ]:
import matplotlib.pyplot as plt

ax = frame["close"].plot(figsize=(12, 4), title="EURUSD, one-second closes (bid)")
ax.set_xlabel("UTC")
plt.show()

## How busy each minute was

`volume` is a tick volume: the number of price updates received in the bar, not a traded volume. Summed per minute, it shows when the market moved.

In [ ]:
per_minute = frame["volume"].resample("1min").sum()
per_minute.plot(kind="bar", figsize=(12, 3), title="Price updates per minute").set_xticklabels([])
plt.show()

## Further

- The live stream of the same pair: [gold_live.ipynb](gold_live.ipynb) shows it on gold.
- Every refusal comes with a code: see the [API reference](https://adona-robot.com/en/reference).
- The client: [adona-market-data](https://github.com/Adona-Robot/market-data-python).